# Minimal COSMOS2020 + FSPS + MAF SBI Tutorial

This notebook is the short, user-facing version of the COSMOS2020 FSPS SBI workflow.
It is intentionally not a performance benchmark.  The goal is to show the normal
CompoSED data flow on real catalog photometry:

1. read one COSMOS2020 photometric subset;
2. declare filters, a backend, priors, and a noise model;
3. bind the backend, priors, observed bands, and likelihood into a `Problem`;
4. simulate from that Problem and train a conditional MAF posterior;
5. sample `q(z, log10_mass | photometry)` for the catalog subset;
6. make simple diagnostics against COSMOS2020/LePhare reference values.

The reference catalog redshifts and masses are **not truth**.  They are used here
only as a sanity check that the pipeline is wired sensibly.

## Setup

For a real run, set these two environment variables before launching the notebook:

```bash
export COSMOS2020_FARMER_FITS=/path/to/cosmos2020_farmer.fits
export SPS_HOME=/path/to/fsps
```

The default sizes below are deliberately small enough for a first interactive run.
Increase `N_TRAIN` and `N_TARGET` once the notebook works end-to-end.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Let the notebook work from either the repo root or the notebooks/ directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "composed").exists() and (REPO_ROOT.parent / "composed").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from composed import DelayedTauSFH, Gaussian, MAF, ParameterSpace, PhotometricContext, Problem, SEDDataset, Simulate, UniformPrior
from composed.backends.fsps import FSPSBackend
from composed.filters import FilterSet
from composed.sbi import simulate_sbi_training_set, train_sbi
from composed.units import MassNormalization

In [ ]:
CATALOG_PATH = Path(os.environ.get("COSMOS2020_FARMER_FITS", "~/Downloads/cosmos2020_farmer.fits")).expanduser()
OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs" / "cosmos2020_fsps_maf_sbi_tutorial"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RNG_SEED = 11
N_TARGET = 512       # catalog galaxies to evaluate in this tutorial
N_TRAIN = 2_000      # FSPS simulations for the tutorial MAF; use 1e5+ for science
N_POST_SAMPLES = 256

# Serial is the most notebook-portable setting.  For large runs, prefer a small
# script with executor="process" and n_workers set to the number of cores you want.
SIMULATION_EXECUTOR = "serial"
N_WORKERS = 1
SIMULATION_BATCH_SIZE = 1

print("catalog:", CATALOG_PATH)
print("output:", OUTPUT_DIR)
print("SPS_HOME:", os.environ.get("SPS_HOME", "<not set>"))

## Read COSMOS2020 Photometry

COSMOS2020 FARMER fluxes are in microJy.  CompoSED photometric likelihoods and
FSPS backend examples use **maggies**, so we convert `uJy -> maggies` immediately.

In [ ]:
BAND_NAMES = ["u", "g", "r", "i", "z", "Y", "J", "H"]
CATALOG_BANDS = [
    "cfht_u",
    "hsc_g",
    "hsc_r",
    "hsc_i",
    "hsc_z",
    "hsc_y",
    "uvista_j",
    "uvista_h",
]
SEDPY_FILTER_NAMES = [
    "cfht_megacam_us_9301",
    "hsc_g",
    "hsc_r",
    "hsc_i",
    "hsc_z",
    "hsc_y",
    "vista_vircam_J",
    "vista_vircam_H",
]
MICROJY_TO_MAGGIES = 10.0 ** (-0.4 * 23.9)


def load_cosmos2020_subset(path: Path, n_target: int, seed: int) -> dict[str, np.ndarray]:
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(
            f"Missing COSMOS2020 catalog: {path}. Set COSMOS2020_FARMER_FITS to the FITS file."
        )

    with fits.open(path, memmap=True) as hdul:
        catalog = hdul[1].data
        n_rows = len(catalog)

        clean = np.isfinite(catalog["lp_zbest"])
        clean &= (catalog["lp_zbest"] > 0.0) & (catalog["lp_zbest"] < 5.0)
        clean &= np.isfinite(catalog["lp_mass_med"])
        clean &= (catalog["lp_mass_med"] > 5.0) & (catalog["lp_mass_med"] < 14.0)
        clean &= catalog["lp_type"] == 0
        clean &= catalog["flag_combined"] == 0

        valid_photometry = np.ones(n_rows, dtype=bool)
        for band in CATALOG_BANDS:
            valid_photometry &= catalog[f"{band}_valid"] > 0
            valid_photometry &= np.isfinite(catalog[f"{band}_flux"])
            valid_photometry &= np.isfinite(catalog[f"{band}_fluxerr"])
            valid_photometry &= catalog[f"{band}_fluxerr"] > 0.0

        snr_r = np.asarray(catalog["hsc_r_flux"] / catalog["hsc_r_fluxerr"], dtype=float)
        snr_i = np.asarray(catalog["hsc_i_flux"] / catalog["hsc_i_fluxerr"], dtype=float)
        selected = clean & valid_photometry & ((snr_r > 5.0) | (snr_i > 5.0))
        eligible_rows = np.flatnonzero(selected)
        n_eligible = eligible_rows.size
        if n_eligible == 0:
            raise RuntimeError("COSMOS2020 selection produced no galaxies.")

        rng = np.random.default_rng(seed)
        rows = rng.choice(eligible_rows, size=min(int(n_target), n_eligible), replace=False)
        rows.sort()

        flux_uJy = np.vstack([np.asarray(catalog[f"{band}_flux"][rows], dtype=float) for band in CATALOG_BANDS]).T
        sigma_uJy = np.vstack([np.asarray(catalog[f"{band}_fluxerr"][rows], dtype=float) for band in CATALOG_BANDS]).T

        return {
            "row_index": rows.astype(np.int64),
            "flux_maggies": (flux_uJy * MICROJY_TO_MAGGIES).astype(np.float32),
            "sigma_maggies": (sigma_uJy * MICROJY_TO_MAGGIES).astype(np.float32),
            "z_ref": np.asarray(catalog["lp_zbest"][rows], dtype=np.float32),
            "log10_mass_ref": np.asarray(catalog["lp_mass_med"][rows], dtype=np.float32),
            "n_catalog_rows": np.array(n_rows),
            "n_eligible": np.array(n_eligible),
        }


catalog = load_cosmos2020_subset(CATALOG_PATH, n_target=N_TARGET, seed=RNG_SEED)
x_obs_flux = catalog["flux_maggies"]
sigma_obs = catalog["sigma_maggies"]

print("catalog flux shape:", x_obs_flux.shape)
print("selected rows:", len(x_obs_flux))
print("z_ref range:", float(catalog["z_ref"].min()), float(catalog["z_ref"].max()))
print("log10_mass_ref range:", float(catalog["log10_mass_ref"].min()), float(catalog["log10_mass_ref"].max()))

## Backend, Priors, and Noise Model

FSPS expects a tabular SFH.  To keep the tutorial prior scalar-valued, the small
adapter below converts `(tau_gyr, age_fraction)` into a delayed-tau tabular SFH.
This is the only model-specific glue in the notebook.

In [ ]:
from sedpy.observate import load_filters

filters = FilterSet(load_filters(SEDPY_FILTER_NAMES), names=BAND_NAMES)

parameter_space = ParameterSpace(
    names=["zred", "log10_mass", "logzsol", "tau_gyr", "age_fraction", "dust2"],
    priors={
        "zred": UniformPrior(0.05, 5.0),
        "log10_mass": UniformPrior(6.0, 13.0),
        "logzsol": UniformPrior(-1.0, 0.2),
        "tau_gyr": UniformPrior(0.2, 5.0),
        "age_fraction": UniformPrior(0.05, 0.95),
        "dust2": UniformPrior(0.0, 2.0),
    },
)

INFER = ["zred", "log10_mass"]


backend = FSPSBackend(
    sfh=DelayedTauSFH(
        age="age_fraction",
        age_kind="fraction_of_universe",
        tau="tau_gyr",
        n_time=64,
    ),
    sp_kwargs={
        "zcontinuous": 1,
        "sfh": 3,
        "add_dust_emission": True,
        "add_neb_emission": True,
        "add_igm_absorption": True,
        "igm_factor": 1.0,
        "compute_vega_mags": False,
    },
    mass_normalization=MassNormalization.PER_SOLAR_MASS,
    default_z_key="zred",
)

# Use the empirical COSMOS2020 median uncertainty per band plus a small relative term.
sigma_abs_maggies = np.median(sigma_obs, axis=0).astype(np.float32)
frac_noise = 0.05


def noise_fn(flux_maggies):
    flux_maggies = np.asarray(flux_maggies, dtype=np.float32)
    return np.sqrt(sigma_abs_maggies**2 + (frac_noise * np.abs(flux_maggies))**2).astype(np.float32)


print("parameter order:", parameter_space.names)
print("inferred parameters:", INFER)
print("mass normalization:", backend.mass_normalization)

## Photometry Features

The stable MAF context stores measured signal-to-noise and log10 uncertainty
for each band.  It therefore conditions on catalog depth explicitly and accepts
negative noisy fluxes without a magnitude transform.

In [ ]:
context = PhotometricContext("snr_logsigma", flux_unit="maggies")
x_obs = context.encode(x_obs_flux, sigma_obs)
theta_ref = np.column_stack([catalog["z_ref"], catalog["log10_mass_ref"]]).astype(np.float32)

print("context features:", x_obs.shape, "finite:", np.isfinite(x_obs).all())
print("feature median/range:", float(np.median(x_obs)), float(np.min(x_obs)), float(np.max(x_obs)))

## Simulate Training Photometry

The observed SED, backend, priors, likelihood, filters, units, and active-band
convention are first bound into one `Problem`. Training simulations are then
drawn from that exact scientific declaration.

In [ ]:
t0 = time.perf_counter()
problem = Problem(
    backend=backend,
    parameters=parameter_space,
    data=SEDDataset(BAND_NAMES, x_obs_flux[0], sigma_obs[0], flux_unit="maggies"),
    likelihood=Gaussian(),
    filters=filters,
)
training = simulate_sbi_training_set(
    problem,
    Simulate(
        n=N_TRAIN,
        noise_fn=noise_fn,
        infer=INFER,
        context=context,
        max_retries=max(100, N_TRAIN // 10),
        executor=SIMULATION_EXECUTOR,
        n_workers=N_WORKERS,
        batch_size=SIMULATION_BATCH_SIZE,
    ),
    rng=RNG_SEED,
)
simulation_seconds = time.perf_counter() - t0

print("training photometry features:", training.x.shape)
print("training theta:", training.theta.shape, training.theta_names)
print("simulation time: {:.1f} s ({:.2f} SED/s)".format(simulation_seconds, N_TRAIN / simulation_seconds))
print("simulation failures:", len(training.metadata["simulate_training_set"]["failures"]))

## Train MAF Posterior

`train_sbi` trains a conditional MAF on the declared `(theta, x)` pairs. For a
real run, increase `N_TRAIN`, epochs, and probably the flow size.

In [ ]:
t0 = time.perf_counter()
maf = train_sbi(
    training,
    MAF(
        hidden_features=128,
        num_transforms=5,
        num_blocks=2,
        learning_rate=3.0e-4,
        device="auto",
        epochs=50,
        batch_size=512,
        verbose=True,
    ),
    seed=RNG_SEED,
)
training_seconds = time.perf_counter() - t0

print("device:", maf.estimator.device)
print("final training loss:", maf.history["train_loss"][-1])
print("training time: {:.1f} s".format(training_seconds))
checkpoint_dir = OUTPUT_DIR / "maf_checkpoint"
maf.save(checkpoint_dir, overwrite=True)
print("checkpoint:", checkpoint_dir)

## Sample the Catalog Posterior

The MAF is amortized: after training, sampling many galaxies is just one batched
neural-network call.  The samples below have shape
`(n_galaxies, n_samples, n_parameters)`.

In [ ]:
t0 = time.perf_counter()
posterior_samples = maf.sample(
    x_obs_flux,
    sigma=sigma_obs,
    input_units="native",
    num_samples=N_POST_SAMPLES,
    batch_size=8192,
    seed=RNG_SEED + 1,
)
sampling_seconds = time.perf_counter() - t0
posterior_median = np.median(posterior_samples, axis=1)

print("posterior samples:", posterior_samples.shape)
print("sampling time: {:.3f} s".format(sampling_seconds))
print("theta-draw rate: {:.1f} / s".format(np.prod(posterior_samples.shape[:2]) / sampling_seconds))

## Quick Diagnostics

These diagnostics compare against COSMOS2020/LePhare reference values.  They are
useful for a first sanity check, but not a simulation-calibration claim because
LePhare estimates are not the generative truth.

In [ ]:
diagnostics = maf.diagnostics(
    posterior_samples,
    theta_ref,
    x_test=x_obs,
    output_dir=OUTPUT_DIR / "diagnostics",
    make_plots=True,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, j, name, label in zip(
    axes,
    [0, 1],
    maf.theta_names,
    ["COSMOS2020 lp_zbest", "COSMOS2020 lp_mass_med"],
):
    ax.hexbin(theta_ref[:, j], posterior_median[:, j], gridsize=35, mincnt=1, cmap="viridis")
    lo = float(np.nanmin(theta_ref[:, j]))
    hi = float(np.nanmax(theta_ref[:, j]))
    ax.plot([lo, hi], [lo, hi], color="white", lw=1.0, alpha=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel(f"CompoSED MAF median {name}")
    ax.grid(alpha=0.2)

fig.suptitle("COSMOS2020 reference values vs CompoSED MAF posterior medians")
fig.savefig(OUTPUT_DIR / "cosmos2020_maf_reference_scatter.png", dpi=160)
plt.show()

mae = np.median(np.abs(posterior_median - theta_ref), axis=0)
for name, value in zip(maf.theta_names, mae):
    print(f"median |posterior median - reference| for {name}: {value:.4g}")
print("diagnostics written to", OUTPUT_DIR / "diagnostics")

## What to Scale Up

For a science or timing run, the minimal changes are:

```python
N_TARGET = 100_000
N_TRAIN = 100_000
N_POST_SAMPLES = 512
```

and run the simulation in a script with process workers rather than a notebook:

```python
simulate_sbi_training_set(problem, Simulate(..., executor="process", n_workers=8, batch_size=256))
```

The main checklist items are the backend, the priors, the noise model, the
photometric context, and the catalog cuts above.